# DockBench — Inference (Colab / Kaggle)

1. `git clone` code từ GitHub  
2. **Tải `results/` từ Google Drive** (trọng số `.pt` — không có trên repo)  
3. Cài molgrid + gninatyper → score **3ERK**

**Runtime:** GPU (khuyến nghị).

## 1. Cấu hình

In [ ]:
# --- GitHub ---
REPO_URL = "https://github.com/nWoWolfpac/KhoaLuanTotNghiep.git"
BRANCH = "main"
CLONE_DIR = "/content/KhoaLuanTotNghiep"  # Colab
# CLONE_DIR = "/kaggle/working/KhoaLuanTotNghiep"  # Kaggle

MODEL_ID = "geoformerdock"
CHECKPOINT = None  # hoặc path tuyệt đối tới best_model.pt
GNINATYPER = None  # None → tìm code_docking/tools/gninatyper

# --- Google Drive: thư mục results (cùng link README.md mục 5) ---
# Cách A (khuyến nghị Colab): tải folder share bằng gdown
DRIVE_FOLDER_ID = "1ItrUVa9PUFoiF_8khIOwb1OWHQURGrAp"
USE_GDOWN_FOLDER = True   # True = tải từ folder ID; False = mount Drive + path bên dưới

# Cách B: sau khi mount Drive, trỏ tới thư mục results đã có trên Drive
RESULTS_DRIVE_PATH = "/content/drive/MyDrive/KhoaLuan/code_docking/results"

## 2. Clone code

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

def sh(cmd, check=True):
    print(">>>", cmd)
    return subprocess.run(cmd, shell=True, check=check)

clone = Path(CLONE_DIR)
if not (clone / ".git").is_dir():
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {CLONE_DIR}")
else:
    sh(f"cd {CLONE_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only", check=False)

if (clone / "code_docking" / "dockbench").is_dir():
    DOCK_ROOT = clone / "code_docking"
elif (clone / "dockbench").is_dir():
    DOCK_ROOT = clone
else:
    raise FileNotFoundError(f"Không thấy dockbench/ trong {clone}")

sys.path.insert(0, str(DOCK_ROOT))
os.chdir(DOCK_ROOT)
print("DOCK_ROOT =", DOCK_ROOT.resolve())

## 3. Load `results/` từ Google Drive

- **Cách A (`USE_GDOWN_FOLDER=True`):** tải folder share (ID trong ô 1) — không cần biết path trên Drive.  
- **Cách B:** `USE_GDOWN_FOLDER=False` → mount Drive, sửa `RESULTS_DRIVE_PATH` cho đúng vị trí `results/`.

Cấu trúc sau khi tải phải có `models/<tên_model>/best_model.pt`.

In [ ]:
def _normalize_results_dir(path: Path) -> Path:
    """Tìm thư mục có con models/ (results/ hoặc lồng sâu hơn)."""
    path = path.resolve()
    for cand in [path, path / "results", path / "code_docking" / "results"]:
        if (cand / "models").is_dir():
            return cand
    raise FileNotFoundError(f"Không thấy models/ dưới {path}")


def load_results_from_drive() -> Path:
    if USE_GDOWN_FOLDER and DRIVE_FOLDER_ID:
        sh("pip install -q gdown", check=False)
        cache = Path("/content/results_drive")
        marker = cache / ".download_ok"
        if not marker.is_file():
            if cache.exists():
                sh(f"rm -rf {cache}")
            cache.mkdir(parents=True, exist_ok=True)
            url = f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}"
            sh(f"gdown --folder {url} -O {cache} --remaining-ok", check=False)
            marker.write_text("ok")
        return _normalize_results_dir(cache)

    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except ImportError:
        raise RuntimeError("Cách B cần Colab, hoặc bật USE_GDOWN_FOLDER=True")

    return _normalize_results_dir(Path(RESULTS_DRIVE_PATH))


RESULTS_DIR = load_results_from_drive()
models_root = RESULTS_DIR / "models"
print("RESULTS_DIR =", RESULTS_DIR)

pts = sorted(models_root.rglob("best_model.pt"))
print(f"Checkpoint .pt: {len(pts)}")
for p in pts[:12]:
    print(f"  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")
if not pts:
    raise FileNotFoundError("Không có best_model.pt — kiểm tra DRIVE_FOLDER_ID hoặc RESULTS_DRIVE_PATH")

## 4. Cài molgrid + PyTorch

In [ ]:
import platform
assert platform.system() == "Linux", "Notebook chỉ chạy trên Colab/Kaggle/Linux."

MF = Path("/usr/local/mambaforge")
if not (MF / "bin/mamba").exists():
    sh("wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh")
    sh("bash /tmp/mf.sh -b -p /usr/local/mambaforge")

sh("/usr/local/mambaforge/bin/mamba install -y -c conda-forge molgrid openbabel 'numpy<2'")
sh("/usr/local/mambaforge/bin/mamba run -n base pip install -q torch py3Dmol pandas")

for pyver in ("3.12", "3.11", "3.10"):
    sp = MF / f"lib/python{pyver}/site-packages"
    if sp.is_dir() and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))

import molgrid
print("molgrid OK")

## 5. gninatyper

In [ ]:
import shutil

def find_gninatyper():
    if GNINATYPER is not None:
        p = Path(GNINATYPER)
        if p.is_file():
            return p
    for cand in [DOCK_ROOT / "tools" / "gninatyper", Path("/content/gninatyper")]:
        if cand.is_file():
            return cand
    w = shutil.which("gninatyper")
    return Path(w) if w else None

gtyper = find_gninatyper()
print("gninatyper:", gtyper or "⚠ chưa có — thêm vào tools/gninatyper hoặc build")

## 6. Bảng metrics

In [ ]:
import pandas as pd
from IPython.display import display
from dockbench.models.registry import BENCHMARK_MODELS, MODEL_DISPLAY_NAMES

def load_summary(model_dir: Path):
    p = model_dir / "summary.json"
    return json.loads(p.read_text()) if p.is_file() else {}

def resolve_ckpt(path_str: str):
    if not path_str:
        return None
    rel = path_str.replace("\\", "/").split("results/")[-1]
    for c in [Path(path_str), RESULTS_DIR / rel, DOCK_ROOT / path_str]:
        if c.is_file():
            return c.resolve()
    return None

rows = []
for mid in BENCHMARK_MODELS:
    s = load_summary(models_root / mid)
    bw = s.get("best_weights", "")
    ck = resolve_ckpt(bw)
    rows.append({
        "model": MODEL_DISPLAY_NAMES.get(mid, mid),
        "C-index": s.get("final_c_index"),
        "BalAcc": s.get("final_bal_acc"),
        "checkpoint": bw,
        ".pt": "✓" if ck else ("N/A" if mid == "equibind" else "✗"),
    })
display(pd.DataFrame(rows))

## 7. Load checkpoint

In [ ]:
import torch
from typing import Tuple
from dockbench.models.registry import build_model, canonical_name
from dockbench.target_normalizer import TargetNormalizer

def load_model(ckpt_path: Path, model_name: str, input_dims: Tuple[int, int, int, int]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    name = canonical_name(model_name or payload.get("model", MODEL_ID))
    summary = load_summary(models_root / name)
    gkw = None
    if name == "geoformerdock":
        gkw = {
            "max_pseudo_atoms": int(summary.get("max_pseudo_atoms", 12)),
            "num_transformer_layers": int(summary.get("num_transformer_layers", 2)),
            "uncertainty": bool(summary.get("geoformer_uncertainty", False)),
        }
    model = build_model(name, input_dims, affinity=True, flex=False, geoformer_kwargs=gkw)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device).eval()
    norm = None
    ns = payload.get("target_normalizer")
    if payload.get("normalize_targets") and ns:
        norm = TargetNormalizer()
        norm.mean, norm.std, norm.fitted = float(ns["mean"]), float(ns["std"]), True
    return model, device, norm, name

summary = load_summary(models_root / MODEL_ID)
ckpt_path = Path(CHECKPOINT) if CHECKPOINT else resolve_ckpt(summary.get("best_weights", ""))
print("Checkpoint:", ckpt_path)
assert ckpt_path and ckpt_path.is_file(), "Không thấy .pt — kiểm tra Drive (mục 3) hoặc đặt CHECKPOINT="

## 8. Score 3ERK

In [ ]:
import urllib.request

PDB_ID, LIGAND = "3ERK", "SB4"
WORK = Path("/content/dockbench_work") / PDB_ID
WORK.mkdir(parents=True, exist_ok=True)

pdb = WORK / f"{PDB_ID}.pdb"
if not pdb.is_file():
    urllib.request.urlretrieve(f"http://files.rcsb.org/download/{PDB_ID}.pdb", pdb)

rec, lig = WORK / "rec.pdb", WORK / "lig.pdb"
rec_lines, lig_lines = [], []
for line in pdb.read_text(errors="replace").splitlines(True):
    if line.startswith("ATOM"):
        rec_lines.append(line)
    elif LIGAND in line:
        lig_lines.append(line)
rec.write_text("".join(rec_lines))
lig.write_text("".join(lig_lines))
print(f"rec={len(rec_lines)} lig={len(lig_lines)} atoms")

def pdb_to_gninatypes(pdb_path: Path, out_path: Path):
    exe = find_gninatyper()
    if not exe:
        raise RuntimeError("Thiếu gninatyper — xem mục 5")
    for cmd in (
        [str(exe), str(pdb_path), "-o", str(out_path)],
        [str(exe), "-r", str(pdb_path), "-o", str(out_path)],
        [str(exe), str(pdb_path), str(out_path)],
    ):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0 and out_path.is_file() and out_path.stat().st_size > 0:
            return
    raise RuntimeError((r.stderr or r.stdout or "gninatyper failed")[:500])

rec_gt, lig_gt = WORK / "rec.gninatypes", WORK / "lig.gninatypes"
pdb_to_gninatypes(rec, rec_gt)
pdb_to_gninatypes(lig, lig_gt)
(WORK / "score.types").write_text(f"1 0.0 {rec_gt.name} {lig_gt.name}\n")
print("gninatypes OK")

In [ ]:
gmaker = molgrid.GridMaker(resolution=0.5, dimension=23.5)
prov = molgrid.ExampleProvider(
    data_root=str(WORK), balanced=False, shuffle=False,
    default_batch_size=1, iteration_scheme=molgrid.IterationScheme.SmallEpoch,
    cache_structs=False,
)
prov.populate(str(WORK / "score.types"))
INPUT_DIMS = tuple(int(x) for x in gmaker.grid_dimensions(prov.num_types()))
print("input_dims =", INPUT_DIMS)

model, device, normalizer, model_name = load_model(ckpt_path, MODEL_ID, INPUT_DIMS)
batch = prov.next_batch(1)
grid = torch.zeros((1,) + INPUT_DIMS, dtype=torch.float32, device=device)
gmaker.forward(batch, grid, random_translation=0.0, random_rotation=False)

with torch.no_grad():
    pose_log, aff = model(grid)

pose_prob = float(torch.exp(pose_log)[0, 1].item())
aff_val = float(aff[0].item())
if normalizer and normalizer.fitted:
    aff_val = float(normalizer.denormalize(aff)[0].item())

print(f"Model: {model_name}")
print(f"Pose P(good): {pose_prob:.4f}  → {int(pose_prob >= 0.5)}")
print(f"Affinity pK: {aff_val:.4f}")

## 9. Xem 3D

In [ ]:
import py3Dmol
view = py3Dmol.view(width=700, height=450)
view.addModel(rec.read_text(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
view.addModel(lig.read_text(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo()
view.show()